## NAME(s):               
## NETID(s): 

# Homework 03: Direct Preference Optimization

This assignment walks you through applying [Direct Preference Optimization (DPO)](https://arxiv.org/pdf/2305.18290) to a pre-trained conversational model using a preference dataset from HuggingFace. You will find a suitable preference dataset, use it to train a model with DPO, and then compare the behavior of the original and DPO-trained models.

This notebook has a lot of text but almost no coding if you choose to use the default dataset. Most of the code is provided for you; your main tasks are (1) finding and preparing a dataset, (2) writing the model comparison code, and (3) reflecting on the results. Read the instructional material carefully before writing any code.

You may work in groups of up to 3 people. When you are done, submit a PDF of this notebook with all cells run.

**Note:** Training is computationally intensive. In this assignment, we keep things manageable by using a small model (`SmolLM2-135M-Instruct`), a small subset of training data, and CPU-only training. Even so, training may take several minutes.

### Import Packages

Run the code cell below to import the packages you will use in this assignment.

In [ ]:
import torch
import pandas as pd
from datasets import Dataset, load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

### Load the Model and Tokenizer

Run the code cell below to load the `HuggingFaceTB/SmolLM2-135M-Instruct` model and its associated tokenizer. This is a small instruction-tuned language model with a standard chat template, meaning it understands conversations formatted with `user` and `assistant` roles.

We set the tokenizer's padding token to the end-of-sequence token, which is required for the DPO trainer.

In [ ]:
sft_model_name = "HuggingFaceTB/SmolLM2-135M-Instruct"
model = AutoModelForCausalLM.from_pretrained(sft_model_name)

tokenizer = AutoTokenizer.from_pretrained(sft_model_name)
tokenizer.pad_token = tokenizer.eos_token

## 1. Find and Load a DPO Dataset from HuggingFace (20 points)

Your first task is to find a preference dataset on the [HuggingFace Hub](https://huggingface.co/datasets) that is suitable for DPO training. A DPO dataset consists of examples where each example has:

- A **prompt** (the input to the model)
- A **chosen** response (the preferred/better response)
- A **rejected** response (the dispreferred/worse response)

You can search for datasets by filtering for tags like `dpo`, `preference`, or `rlhf` on the HuggingFace Hub. Here are some datasets that would work well for this assignment:

| Dataset | Size | Notes |
|---------|------|-------|
| `Anthropic/hh-rlhf` | ~170k | Classic helpfulness/harmlessness preferences. Requires parsing conversation format. |
| `argilla/dpo-mix-7k` | ~7k | Curated mix of DPO data. Already in prompt/chosen/rejected format. |
| `jondurbin/truthy-dpo-v0.1` | ~1.4k | Focuses on truthful responses. Smaller and easier to work with. |

**Important:** You should **only download a small subset of the dataset** (50-100 examples). Training on more data will take too long on CPU. You can do this by slicing the split string when calling `load_dataset`:

```python
# This loads only the first 50 examples from the training split
dataset = load_dataset("dataset-name", split="train[:50]")
```

In the cell below, load your chosen dataset. You are free to use any DPO/preference dataset from HuggingFace, not just the ones listed above.

**Warning:** Preference datasets can contain disturbing, offensive, obscene, or otherwise unpleasant content. If you want to avoid this, `truthy-dpo-v0.1` may be a good choice.

In [ ]:
# Load a subset of the dataset from HuggingFace
# Your code here

# using jondurbin/truthy-dpo-v0.1 as an example
raw_dataset = load_dataset("jondurbin/truthy-dpo-v0.1", split="train[:100]")

### Inspect Your Dataset

Before preparing the data, you should inspect it to understand its structure. Print the column names, the number of examples, and look at a few rows. This will help you determine what (if any) reformatting is needed in the next step.

**Warning**: Preference datasets can contain disturbing, obscene, or otherwise unpleasant content. If you want to avoid this, `truthy-dpo-v0.1` may be a better choice. 

In [ ]:
# Inspect the dataset structure
# Your code here 

print(f"Number of examples: {len(raw_dataset)}")
print(f"Column names: {raw_dataset.column_names}")
print(f"\nFirst example:")
raw_dataset[0]

## 2. Prepare the Data for DPO Training (20 points)

The `DPOTrainer` expects each example in the dataset to have exactly three columns: `prompt`, `chosen`, and `rejected`.

Since our model (`SmolLM2-135M-Instruct`) uses a chat template, we should format the data in **conversational format**: each column should be a list of message dictionaries with `role` and `content` keys, rather than plain strings. This ensures the DPO trainer applies the model's chat template during tokenization, so the training format matches how the model was originally fine-tuned.

For example:
```python
{
    "prompt": [{"role": "user", "content": "What is the capital of France?"}],
    "chosen": [{"role": "assistant", "content": "The capital of France is Paris."}],
    "rejected": [{"role": "assistant", "content": "France's capital is London."}],
}
```

Depending on the dataset you chose, you may also need to:
- **Rename columns** (e.g., if your dataset uses `question` instead of `prompt`)
- **Drop extra columns** that aren't needed

In the cell below, write code to transform your dataset into the conversational format described above.

In [ ]:
# Transform your dataset into the required conversational format
# Your code here

def to_chat_format(example):
    return {
        "prompt": [{"role": "user", "content": example["prompt"]}],
        "chosen": [{"role": "assistant", "content": example["chosen"]}],
        "rejected": [{"role": "assistant", "content": example["rejected"]}],
    }

dataset = raw_dataset.map(to_chat_format, remove_columns=["id", "source", "system"])

print(f"Columns: {dataset.column_names}")
print(f"\nExample after formatting:")
dataset[0]

### Split the Data

Split your prepared dataset into training and test sets. We'll use 90% for training and 10% for testing.

In [ ]:
# Split into 90% train and 10% test
split_dataset = dataset.train_test_split(test_size=0.1, seed=42)

train_dataset = split_dataset["train"]
test_dataset = split_dataset["test"]

print(f"Training examples: {len(train_dataset)}")
print(f"Test examples: {len(test_dataset)}")
print(f"\nFirst training example: {train_dataset[0]}")

## 3. Train a Model Using DPO (20 points)

Now train the model using DPO. The cell below sets up the training configuration and runs training. You do not need to modify this code, but you should read through it and understand what each parameter does.

A few parameters worth noting:
- `beta=0.1`: Controls how much the DPO-trained model can diverge from the reference model. Lower values allow more divergence.
- `max_length=256`: Truncate sequences to keep training fast.
- `use_cpu=True`: We train on CPU since we're working with a small model and dataset.

**Training may take several minutes. This is normal.**

In [ ]:
from trl import DPOTrainer, DPOConfig

# Set up training configuration
training_args = DPOConfig(
    output_dir="./dpo_model",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    learning_rate=5e-5,
    save_strategy="epoch",
    logging_steps=5,
    remove_unused_columns=False,
    report_to=[],
    bf16=False,
    fp16=False,
    use_cpu=True,
    max_length=256,
    beta=0.1,
)

# Create DPO trainer
print("Setting up DPO trainer...")
trainer = DPOTrainer(
    model=model,
    train_dataset=train_dataset,
    args=training_args,
)

# Train the model
print("Training with DPO on CPU...")
trainer.train()
trainer.save_model()

print("Training complete! Model saved to ./dpo_model")

## 4. Compare Fine-Tuned and DPO-Trained Models (20 points)

Now compare the outputs of the original instruction-tuned model (`SmolLM2-135M-Instruct`) and your DPO-trained model. Use the `generate` function that takes a model and a prompt string and returns the model's generated text. Use the tokenizer's `apply_chat_template` method to format the prompt as a chat message before passing it to the model.

Then, use prompts from your test set to generate outputs from both models and display them side by side. 

For each test prompt, display:
- The prompt
- The SFT model's output
- The DPO model's output
- The chosen and rejected responses from the dataset (for reference)

In [ ]:
# Write a generate function and compare both models on test prompts

def generate(model, prompt):
    """Generate text from a model given a prompt, using the chat template."""
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(messages, return_tensors="pt", return_dict=True, add_generation_prompt=True)
    input_len = inputs["input_ids"].shape[-1]
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    # Decode only the newly generated tokens (skip the prompt)
    return tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)


# Load fresh copies of both models
dpo_model = AutoModelForCausalLM.from_pretrained("./dpo_model")
sft_model = AutoModelForCausalLM.from_pretrained(sft_model_name)

# Compare on test examples
num_examples = min(5, len(test_dataset))
for i in range(num_examples):
    prompt = test_dataset[i]["prompt"][0]["content"]  # extract text from chat format

    sft_output = generate(sft_model, prompt)
    dpo_output = generate(dpo_model, prompt)

    print(f"\n=== Example {i+1} ===")
    print(f"Prompt:\n{prompt}\n")

    print("SFT MODEL OUTPUT:")
    print(sft_output.strip())

    print("\nDPO MODEL OUTPUT:")
    print(dpo_output.strip())

    print("\nCHOSEN (Reference):")
    print(test_dataset[i]["chosen"][0]["content"])

    print("\nREJECTED (Reference):")
    print(test_dataset[i]["rejected"][0]["content"])

    print("-" * 80)

## 5. Reflection (20 points)

Write a brief response to each of the following questions.

### 1. How did the DPO-trained model's outputs differ from the original fine-tuned model? Were the differences consistent with the preference data you used?

*Your response here*

### 2. At a conceptual level, how does the approach we used with DPO here differ from what we did with GRPO in lecture? 

*Your response here*

### 3. If you wanted to meaningfully improve the DPO-trained model's quality, what would you change? Consider factors like dataset choice, dataset size, model size, and training hyperparameters.

*Your response here*